In [ ]:

# Instalación limpia estándar sin forzar versiones antiguas
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-j55wuxw7/unsloth_51383c8f243f49849bac0ab0e4881cf9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-j55wuxw7/unsloth_51383c8f243f49849bac0ab0e4881cf9
  Resolved https://github.com/unslothai/unsloth.git to commit 85692f1c1c0c19792b27f9e41d603bff776342ff
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

print("Descargando modelo base...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Descargando modelo base...
==((====))==  Unsloth 2026.5.10: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

In [ ]:
print("Aplicando configuración LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rango sugerido para buen aprendizaje
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, # Unsloth recomienda 0 para máxima velocidad
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Ahorro de memoria extremo
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Aplicando configuración LoRA...


Unsloth 2026.5.10 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Forzamos el uso de la plantilla de chat de Llama 3
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

print("Cargando y procesando dataset.jsonl...")
dataset = load_dataset("json", data_files="tutor_violencia_mexico.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Dataset listo: {len(dataset)} ejemplos.")

Cargando y procesando dataset.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Dataset listo: 60 ejemplos.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Iniciando motor de entrenamiento...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Puede acelerar el entrenamiento en secuencias cortas
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Ajusta esto a la cantidad de pasos que desees (ej. 60 a 100)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("\n¡Entrenamiento completado!")

Iniciando motor de entrenamiento...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60 | Num Epochs = 8 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.105614
2,3.248072
3,3.015700
4,2.860917
5,2.529127
6,2.438853
7,2.102252
8,1.867430
9,1.670376
10,1.391916


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.



¡Entrenamiento completado!


In [ ]:
print("Exportando modelo a formato GGUF para Ollama...")

# quantization_method = "q4_k_m" comprime el modelo para que ocupe ~2.2 GB
# ideal para correr en tu laptop con la RTX 3050 sin sudar.
model.save_pretrained_gguf("tutor_mexico", tokenizer, quantization_method = "q4_k_m")

print("¡Exportación finalizada! Ve a la carpeta de archivos a la izquierda y descarga 'tutor_mexico-unsloth-Q4_K_M.gguf'")

Exportando modelo a formato GGUF para Ollama...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in tutor_mexico/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:32<00:00, 76.46s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:42<00:00, 51.17s/it]


Unsloth: Merge process complete. Saved to `/content/tutor_mexico`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['tutor_mexico_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model fil